In [ ]:
# Cell 1: load historical log output + coordinated sweep progress.
from pathlib import Path
from collections import deque, Counter
import json, time

# ============================ EDIT THIS PER MACHINE ============================
VM_NAME = "vmA"          # <-- which VM's log to follow (Pod_1 -> VM1, etc.)
# =============================================================================

REPO = Path('/workspace/stable-query-latent')
LOG = Path('/workspace/stable_query_latent_logs') / f'pipeline_{VM_NAME}.log'   # per-VM
OUT_DIR = REPO / 'VICReg_review/heads/cloud_full_sweep_a100'                    # SHARED
EMBED_MANIFEST = REPO / 'game_review_data/embedding_h5.h5.incloud_manifest.json'
TEXT_MANIFEST = REPO / 'game_review_data/build_new_gamedata/text_h5.h5.manifest.json'


def tail(path=LOG, lines=200):
    path = Path(path)
    print(f'log: {path}')
    if not path.exists():
        print('missing log file (nothing written yet for this VM)')
        return
    with path.open('r', encoding='utf-8', errors='replace') as f:
        for line in deque(f, maxlen=lines):
            print(line, end='')


def show_manifest(path):
    path = Path(path)
    print(f'\n=== {path.name} ===')
    if not path.exists():
        print('missing')
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        print('bad json:', exc)
        return
    for key in ['status', 'updated_at', 'finished_at', 'error']:
        if key in data:
            print(f'{key}: {data[key]}')


def show_coordination(out_dir=OUT_DIR):
    # Global progress from the SHARED coordination markers (checkpoint/done.json =
    # done; failed.json = failed; status.json = in flight). Every VM's work shows up
    # here since they share this out_dir. (The ledger is now machine-local scratch.)
    out_dir = Path(out_dir)
    print(f'\n=== coordinated sweep @ {out_dir.name} ===')
    if not out_dir.exists():
        print('out_dir not created yet')
        return
    done = failed = inflight = 0
    inflight_by = Counter()
    for d in out_dir.iterdir():
        if not d.is_dir() or d.name == 'VM_parallel':
            continue
        if (d / 'vicreg_review_h5_latest.pt').exists() or (d / 'done.json').exists():
            done += 1
        elif (d / 'failed.json').exists():
            failed += 1
        elif (d / 'status.json').exists():
            inflight += 1
            try:
                inflight_by[json.loads((d / 'status.json').read_text()).get('vm', '?')] += 1
            except Exception:
                pass
    print(f'done={done}  failed={failed}  in-flight={inflight}   in-flight by: {dict(inflight_by)}')
    vmp = out_dir / 'VM_parallel'
    for f in sorted(vmp.glob('*.json')) if vmp.exists() else []:
        try:
            rec = json.loads(f.read_text())
        except Exception:
            rec = {}
        exp = rec.get('expiry')
        live = '' if exp is None else ('(alive)' if exp > time.time() else '(lease expired)')
        print(f"  vm {rec.get('vm', f.stem):16} {live}  {rec.get('info', {})}")


tail(lines=200)
show_manifest(TEXT_MANIFEST)
show_manifest(EMBED_MANIFEST)
show_coordination()
HISTORY_END = LOG.stat().st_size if LOG.exists() else 0

In [ ]:
# Cell 2: start realtime latest log output in the background.
# Re-run this cell to restart the log watcher. It does not stop the training job.
import threading, time
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets


START_OFFSET = globals().get('HISTORY_END', None)
WATCHERS = globals().setdefault('WATCHERS', {})


def stop_watcher(name):
    item = WATCHERS.get(name)
    if item:
        item['stop'].set()
        print(f'stopping {name} watcher')


def read_new_text(path, start_offset):
    if not path.exists():
        return start_offset, ''
    size = path.stat().st_size
    if start_offset is None or start_offset > size:
        start_offset = size
    with path.open('rb') as f:
        f.seek(start_offset)
        data = f.read()
        end_offset = f.tell()
    text = data.decode('utf-8', errors='replace')
    return end_offset, text


def follow(output, stop_event, path=LOG, interval=5, start_offset=START_OFFSET):
    path = Path(path)
    last_offset = start_offset
    last_update = None
    with output:
        print(f'{time.strftime("%Y-%m-%d %H:%M:%S")} | {path}')
        print('-' * 100)
        print('waiting for new log lines after historical output')
    while not stop_event.is_set():
        if path.exists():
            last_offset, new_text = read_new_text(path, last_offset)
            if new_text:
                last_update = time.strftime('%Y-%m-%d %H:%M:%S')
                with output:
                    print('-' * 100)
                    print(f'new output received at {last_update}')
                    print('-' * 100)
                    print(new_text, end='' if new_text.endswith('\n') else '\n')
        else:
            if last_update is None:
                with output:
                    print(f'{time.strftime("%Y-%m-%d %H:%M:%S")} missing log file: {path}')
        stop_event.wait(interval)


stop_watcher('log')
log_output = widgets.Output(layout={'border': '1px solid #ddd', 'height': '520px', 'overflow_y': 'auto'})
display(log_output)
log_stop = threading.Event()
log_thread = threading.Thread(target=follow, args=(log_output, log_stop), daemon=True)
WATCHERS['log'] = {'stop': log_stop, 'thread': log_thread, 'output': log_output}
log_thread.start()
print('log watcher started in background')


In [ ]:
# Cell 3: realtime dashboard in the background (system + sweep progress).
# Renders ONE fixed-size character-art panel that refreshes IN PLACE
# (clear_output per tick) -- no appended log lines. Re-run this cell to
# restart the watcher; it does not stop the training job.
import json, subprocess, threading, time
from pathlib import Path
from IPython.display import clear_output, display
import ipywidgets as widgets

try:
    import psutil
except ImportError:
    psutil = None
    print('psutil is missing. Run: pip install psutil')

WATCHERS = globals().setdefault('WATCHERS', {})
W = 76                    # dashboard inner width (chars)
TICK_SECONDS = 5          # system stats refresh
SWEEP_SCAN_SECONDS = 30   # shared-FS sweep rescan (dir walk is slower than psutil)


def stop_watcher(name):
    item = WATCHERS.get(name)
    if item:
        item['stop'].set()
        print(f'stopping {name} watcher')


def _read_int(path):
    try:
        text = Path(path).read_text().strip()
        if text == 'max':
            return None
        return int(text)
    except Exception:
        return None


def _f(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def get_memory_status():
    limit = _read_int('/sys/fs/cgroup/memory.max')
    used = _read_int('/sys/fs/cgroup/memory.current')
    if limit is None or used is None:
        limit = _read_int('/sys/fs/cgroup/memory/memory.limit_in_bytes')
        used = _read_int('/sys/fs/cgroup/memory/memory.usage_in_bytes')
    if limit and used and limit < 10**18:
        return used / limit * 100, used / 1024**3, limit / 1024**3, 'cgroup'
    if psutil is None:
        return None, None, None, 'unavailable'
    vm = psutil.virtual_memory()
    return vm.percent, vm.used / 1024**3, vm.total / 1024**3, 'host'


def get_gpu_stats():
    """Structured per-GPU stats for the dashboard; (rows, error_text)."""
    try:
        out = subprocess.run(
            [
                'nvidia-smi',
                '--query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw',
                '--format=csv,noheader,nounits',
            ],
            capture_output=True, text=True, timeout=3,
        ).stdout.strip()
        if not out:
            return [], 'no nvidia-smi output'
        gpus = []
        for line in out.splitlines():
            parts = [part.strip() for part in line.split(',')]
            if len(parts) >= 7:
                gpus.append({
                    'index': parts[0], 'name': parts[1], 'util': _f(parts[2]),
                    'mem_used': _f(parts[3]), 'mem_total': _f(parts[4]),
                    'temp': _f(parts[5]), 'power': _f(parts[6]),
                })
        return gpus, None
    except Exception as exc:
        return [], str(exc)


# Grid size for the sweep progress denominator (best effort; falls back to the
# number of combo dirs seen so far if the sweep config can't be loaded).
try:
    import sys
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))
    from VICReg_review.sweep.config import SweepConfig
    GRID_TOTAL = len(list(SweepConfig.load(str(REPO / 'VICReg_review/sweep/sweep.yaml')).iter_combos()))
except Exception:
    GRID_TOTAL = None


def scan_sweep(out_dir=OUT_DIR):
    """Light version of check_paralle's buckets, from marker files only. One
    manifest read happens only for ckpt dirs missing done.json (migration rule)."""
    out_dir = Path(out_dir)
    if not out_dir.exists():
        return None
    stats = {'done': 0, 'failed': 0, 'inflight': 0, 'ckpt': 0,
             'inflight_by': {}, 'vms_alive': [], 'vms_total': 0}
    for d in out_dir.iterdir():
        if not d.is_dir() or d.name == 'VM_parallel':
            continue
        is_done = (d / 'done.json').exists()
        if not is_done and (d / 'vicreg_review_h5_latest.pt').exists():
            try:
                payload = json.loads((d / 'vicreg_review_h5_manifest.json').read_text(encoding='utf-8'))
                is_done = payload.get('status') == 'done'
            except Exception:
                pass
        if is_done:
            stats['done'] += 1
        elif (d / 'failed.json').exists():
            stats['failed'] += 1
        elif (d / 'status.json').exists():
            stats['inflight'] += 1
            try:
                vm = json.loads((d / 'status.json').read_text(encoding='utf-8')).get('vm', '?')
                stats['inflight_by'][vm] = stats['inflight_by'].get(vm, 0) + 1
            except Exception:
                pass
        elif (d / 'vicreg_review_h5_latest.pt').exists():
            stats['ckpt'] += 1
    vmp = out_dir / 'VM_parallel'
    now = time.time()
    for f in (sorted(vmp.glob('*.json')) if vmp.exists() else []):
        stats['vms_total'] += 1
        try:
            rec = json.loads(f.read_text(encoding='utf-8'))
        except Exception:
            continue
        exp = rec.get('expiry')
        if exp is not None and exp > now:
            stats['vms_alive'].append(rec.get('vm', f.stem))
    return stats


# ---- character-art rendering -----------------------------------------------

def bar(frac, width=24):
    frac = 0.0 if frac is None else min(max(float(frac), 0.0), 1.0)
    filled = int(round(frac * width))
    return '█' * filled + '░' * (width - filled)


def box_top(title, stamp):
    head = f'─ {title} '
    tail = f' {stamp} ─'
    return '┌' + head + '─' * max(0, W - len(head) - len(tail)) + tail + '┐'


def box_sep(title=''):
    head = f'─ {title} ' if title else ''
    return '├' + head + '─' * max(0, W - len(head)) + '┤'


def box_row(text=''):
    return '│ ' + text[:W - 2].ljust(W - 2) + ' │'


def box_bottom():
    return '└' + '─' * W + '┘'


def render(gpus, gpu_err, cpu, ram, disk_rw, sweep, sweep_stamp):
    lines = [box_top(f'{VM_NAME} monitor', time.strftime('%Y-%m-%d %H:%M:%S'))]

    if gpu_err:
        lines.append(box_row(f'GPU    n/a ({gpu_err})'))
    for g in gpus:
        util = g['util']
        vram_frac = (g['mem_used'] / g['mem_total']) if g['mem_used'] is not None and g['mem_total'] else None
        temp = f"{g['temp']:.0f}C" if g['temp'] is not None else '?C'
        power = f"{g['power']:.0f}W" if g['power'] is not None else '?W'
        lines.append(box_row(f"GPU{g['index']}  {g['name']}"))
        lines.append(box_row(f"  util [{bar((util or 0) / 100)}] {util if util is not None else 0:3.0f}%   {temp}  {power}"))
        vram_txt = (f"{g['mem_used']:.1f}/{g['mem_total']:.1f} GiB"
                    if vram_frac is not None else 'n/a')
        lines.append(box_row(f"  vram [{bar(vram_frac)}] {vram_txt}"))

    lines.append(box_sep('host'))
    if cpu is not None:
        lines.append(box_row(f"CPU    [{bar(cpu / 100)}] {cpu:3.0f}%"))
    else:
        lines.append(box_row('CPU    n/a (psutil missing)'))
    ram_pct, ram_used, ram_total, ram_source = ram
    if ram_pct is not None:
        lines.append(box_row(f"RAM    [{bar(ram_pct / 100)}] {ram_pct:3.0f}%  "
                             f"{ram_used:.1f}/{ram_total:.1f} GiB ({ram_source})"))
    else:
        lines.append(box_row('RAM    n/a'))
    if disk_rw is not None:
        lines.append(box_row(f"DISK   read {disk_rw[0]:8.1f} MB/s   write {disk_rw[1]:8.1f} MB/s"))
    else:
        lines.append(box_row('DISK   n/a'))

    lines.append(box_sep(f'sweep @ {Path(OUT_DIR).name}  (rescan every {SWEEP_SCAN_SECONDS}s, last {sweep_stamp})'))
    if sweep is None:
        lines.append(box_row('out_dir not created yet'))
    else:
        counted = sweep['done'] + sweep['failed'] + sweep['inflight'] + sweep['ckpt']
        denom = GRID_TOTAL or counted
        pct = 100.0 * sweep['done'] / denom if denom else 0.0
        denom_txt = str(denom) if GRID_TOTAL else f'>={denom}'
        lines.append(box_row(f"done   [{bar(sweep['done'] / denom if denom else 0)}] "
                             f"{sweep['done']}/{denom_txt} ({pct:.1f}%)"))
        by = '  '.join(f'{vm}={n}' for vm, n in sorted(sweep['inflight_by'].items())) or '-'
        lines.append(box_row(f"live   {sweep['inflight']:3d}  ({by})"))
        lines.append(box_row(f"failed {sweep['failed']:3d}   ckpt-only {sweep['ckpt']:3d}"
                             '   (details: check_paralle.ipynb)'))
        alive = ', '.join(sweep['vms_alive']) or '-'
        lines.append(box_row(f"VMs    alive {len(sweep['vms_alive'])}/{sweep['vms_total']}: {alive}"))
    lines.append(box_bottom())
    return '\n'.join(lines)


def dashboard(output, stop_event, interval=TICK_SECONDS):
    last_disk = psutil.disk_io_counters() if psutil else None
    last_t = time.time()
    if psutil:
        psutil.cpu_percent(interval=None)
    sweep, last_scan, sweep_stamp = None, 0.0, 'never'
    while not stop_event.is_set():
        gpus, gpu_err = get_gpu_stats()
        cpu = psutil.cpu_percent(interval=None) if psutil else None
        ram = get_memory_status()
        disk_rw = None
        if psutil:
            now_disk = psutil.disk_io_counters()
            now_t = time.time()
            dt = max(now_t - last_t, 1e-6)
            disk_rw = ((now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt,
                       (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt)
            last_disk, last_t = now_disk, now_t
        if time.time() - last_scan >= SWEEP_SCAN_SECONDS:
            try:
                sweep = scan_sweep()
                sweep_stamp = time.strftime('%H:%M:%S')
            except Exception as exc:
                sweep_stamp = f'scan error: {exc}'
            last_scan = time.time()
        frame = render(gpus, gpu_err, cpu, ram, disk_rw, sweep, sweep_stamp)
        with output:
            clear_output(wait=True)
            print(frame)
        stop_event.wait(interval)


stop_watcher('system')
system_output = widgets.Output(layout={'border': '1px solid #ddd'})
display(system_output)
system_stop = threading.Event()
system_thread = threading.Thread(target=dashboard, args=(system_output, system_stop), daemon=True)
WATCHERS['system'] = {'stop': system_stop, 'thread': system_thread, 'output': system_output}
system_thread.start()
print('dashboard watcher started in background (stop via Cell 4)')


In [ ]:
# Cell 4: stop background realtime watchers.
WATCHERS = globals().get('WATCHERS', {})
for name, item in list(WATCHERS.items()):
    item['stop'].set()
    print(f'stopped {name} watcher')
